# VERIMIND: manual NLI model training

This notebook trains the lightweight TF-IDF/MLP model, not the optional pretrained DeBERTa model. Run these cells from top to bottom in VS Code with the project's Python 3.11 environment selected. Edit `EPOCHS`, `PATIENCE`, or `EXTRA_EXAMPLES`, then rerun the cells to try another training run. The current built-in dataset has only 42 synthetic examples, so validation scores are for practice and are not real-world accuracy. Never add held-out `dev` or `test` evaluation cases to training.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "verimind").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from verimind.data import build_synthetic_dataset
from verimind.nli_model import NLIModel

LABELS = ["entailment", "contradiction", "neutral"]
print(f"Project: {ROOT}")

## 1. Set training options and add labeled examples

Examples must be `(claim, evidence, label)` tuples. Use only evidence/labels that you are allowed to train on. Labels are `entailment`, `contradiction`, or `neutral`.

In [ ]:
EPOCHS = 100
PATIENCE = 12
VALIDATION_FRACTION = 0.25
SEED = 42

EXTRA_EXAMPLES = [
    # ("The claim to verify.", "A labeled evidence sentence.", "entailment"),
    # ("The claim conflicts with evidence.", "The contradictory evidence.", "contradiction"),
    # ("A claim without related evidence.", "An unrelated evidence sentence.", "neutral"),
]

dataset = build_synthetic_dataset()
pairs = list(dataset["pairs"])
labels = list(dataset["labels"])
for claim, evidence, label in EXTRA_EXAMPLES:
    if label not in LABELS:
        raise ValueError(f"Unsupported label: {label}")
    pairs.append((claim, evidence))
    labels.append(label)

print(f"Total examples: {len(pairs)}")
for label in LABELS:
    print(f"{label}: {labels.count(label)}")

## 2. Make a stratified train/validation split

In [ ]:
train_pairs, validation_pairs, train_labels, validation_labels = train_test_split(
    pairs,
    labels,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=labels,
)
print(f"Training examples: {len(train_pairs)}")
print(f"Validation examples: {len(validation_pairs)}")

## 3. Train and inspect every epoch

The classifier is updated one epoch at a time internally. The history table below shows training and validation accuracy; early stopping keeps the checkpoint with the best validation accuracy.

In [ ]:
model = NLIModel(random_state=SEED)
history = model.fit_with_validation(
    train_pairs,
    train_labels,
    validation_pairs,
    validation_labels,
    epochs=EPOCHS,
    patience=PATIENCE,
)
best_row = max(history, key=lambda row: row["validation_accuracy"])
print(f"Best epoch: {best_row['epoch']}")
print(f"Best validation accuracy: {best_row['validation_accuracy']:.1%}\n")
print("Epoch | Train accuracy | Validation accuracy")
for row in history:
    print(f"{row['epoch']:>5} | {row['train_accuracy']:>14.1%} | {row['validation_accuracy']:>20.1%}")

## 4. Evaluate on the held-out validation split

This is only the small validation split created above. For the project's separate evaluation, tune with `python eval.py --split dev` and reserve `python eval.py --split test` for a final check.

In [ ]:
validation_predictions = model.predict(validation_pairs)
print(f"Validation accuracy: {accuracy_score(validation_labels, validation_predictions):.1%}")
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0,
))
matrix = confusion_matrix(
    validation_labels,
    validation_predictions,
    labels=LABELS,
)
print(f"Confusion matrix (rows=true, columns=predicted; labels={LABELS}):")
print(matrix)

## 5. Save the selected model

The API loads this artifact when it starts. Restart the API after saving.

In [ ]:
MODEL_PATH = ROOT / "artifacts" / "nli_model.pkl"
model.save(MODEL_PATH)
print(f"Saved model: {MODEL_PATH}")